# KoGPT2 파인튜닝

HuggingFace 한국어 instruction 데이터셋(KoAlpaca)으로 skt/kogpt2-base-v2 를 파인튜닝한다.

런타임 → 런타임 유형 변경 → GPU(T4) 설정 후 실행.

In [ ]:


!pip -q install "transformers==4.57.6" "tokenizers==0.22.2" datasets accelerate huggingface_hub
print("설치 완료 — 만약 4번 셀 토크나이저 검사에서 실패하면 [런타임 → 세션 다시 시작] 후 다시 [모두 실행] 하세요.")

In [ ]:

import torch
print('GPU 사용 가능:', torch.cuda.is_available())
print('장치:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:

from datasets import load_dataset

dataset = load_dataset('beomi/KoAlpaca-v1.1a', split='train')
print(dataset)
print('예시:', dataset[0])

dataset = dataset.select(range(3000))
print('사용할 샘플 수:', len(dataset))

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'skt/kogpt2-base-v2'
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>',
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print('파라미터 수:', model.num_parameters())

_test = '파이썬이 뭐야?'
_round = tokenizer.decode(tokenizer.encode(_test), skip_special_tokens=True)
print('토크나이저 한국어 테스트:', repr(_round))
assert _test.replace(' ', '') in _round.replace(' ', ''), (
    '❌ 토크나이저가 한국어를 못 읽습니다(바이트 폴백). '
    '1번 셀로 버전을 고정한 뒤 [런타임 → 세션 다시 시작 및 모두 실행] 하세요.'
)
print('✅ 토크나이저 정상 — 학습 진행 가능')

In [ ]:


PROMPT = '### 질문: {q}\n### 답변: {a}'
MAX_LEN = 256

def format_example(ex):
    text = PROMPT.format(q=ex['instruction'].strip(), a=str(ex['output']).strip())
    text = text + tokenizer.eos_token
    
    return tokenizer(text, truncation=True, max_length=MAX_LEN)

tokenized = dataset.map(format_example, remove_columns=dataset.column_names)
print(tokenized)

In [ ]:

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

args = TrainingArguments(
    output_dir='kogpt2-finetuned',
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=3e-5,
    fp16=False,            
    warmup_ratio=0.1,      
    max_grad_norm=1.0,     
    logging_steps=50,
    save_strategy='epoch',
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
trainer.train()



In [ ]:

save_dir = 'kogpt2-finetuned'
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('저장 완료:', save_dir)

In [ ]:

import torch

def ask(question, max_new_tokens=80):
    prompt = '### 질문: ' + question + '\n### 답변:'
    ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                         top_p=0.92, temperature=0.7, repetition_penalty=1.3,
                         no_repeat_ngram_size=3, pad_token_id=tokenizer.pad_token_id,
                         eos_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].split('### 질문')[0].strip()

for q in ['파이썬이 뭐야?', '건강하게 사는 방법 알려줘', '딥러닝과 머신러닝의 차이는?']:
    print('Q:', q)
    print('A:', ask(q))
    print()

## 학습된 모델 업로드

HuggingFace Hub 에 올려 모델 이름으로 불러올 수 있게 한다.

In [ ]:
from huggingface_hub import login, create_repo

login(token='hf_본인_WRITE_토큰')
REPO = 'jjminu/kogpt2-koalpaca'
create_repo(REPO, exist_ok=True)
model.push_to_hub(REPO)
tokenizer.push_to_hub(REPO)
print('업로드 완료:', REPO)

In [ ]:

!zip -r kogpt2-finetuned.zip kogpt2-finetuned
from google.colab import files
files.download('kogpt2-finetuned.zip')
